In [1]:
import pandas as pd
import numpy as np
from datetime import datetime

In [2]:
pd.set_option("display.max_columns", None)

In [3]:
vacantes = pd.read_excel(r'C:\Users\jober\Downloads\Vacantes_publicadas_2024-2025.xlsx', sheet_name="Vacantes")

In [4]:
# Typing column the names 
vacantes.columns = vacantes.columns.str.lower()
vacantes.columns = vacantes.columns.str.replace(" ","_")

In [5]:
vacantes = vacantes.drop('empre_reg', axis=1)

In [6]:
# Cleaning NaN data. Data as 'nan' are not actual "NaN":
for column in vacantes.columns:
    vacantes[column] = vacantes[column].replace('nan', pd.NA)
    vacantes[column] = vacantes[column].replace('', pd.NA)

In [7]:
vacantes.columns

Index(['código_proceso', 'nombre_vacante', 'cargo', '#_postulados', 'empresa',
       'tipodocumentoempresa', 'numerodocumentoempresa', 'fecha_registro',
       'fecha_vencimiento', 'estado_actual', 'tipo_de_vacante',
       'programa_de_gobierno', 'ubicación', 'discapacidad',
       'puestos_de_trabajo', 'tipo_de_contrato', 'agente_aprobó',
       'fecha_publicación', 'pertenece_hidrocarburos',
       'tipo_de_proyecto_hidrocarburos', 'requiere_mano_de_obra_calificada',
       'mes', 'año', 'punto_atención', 'país'],
      dtype='object')

In [8]:
vacantes.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1428 entries, 0 to 1427
Data columns (total 25 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   código_proceso                    1428 non-null   object 
 1   nombre_vacante                    1428 non-null   object 
 2   cargo                             1428 non-null   object 
 3   #_postulados                      1428 non-null   int64  
 4   empresa                           1428 non-null   object 
 5   tipodocumentoempresa              1428 non-null   object 
 6   numerodocumentoempresa            1428 non-null   object 
 7   fecha_registro                    1428 non-null   object 
 8   fecha_vencimiento                 1428 non-null   object 
 9   estado_actual                     1428 non-null   object 
 10  tipo_de_vacante                   1428 non-null   object 
 11  programa_de_gobierno              0 non-null      float64
 12  ubicac

In [9]:
for col in vacantes.columns:
    if vacantes[col].dtype == 'object':
        vacantes[col] = vacantes[col].astype(str)
        vacantes[column] = [str(i).lower() for i in vacantes[column]]
        vacantes[column] = [str(i).strip() for i in vacantes[column]]

In [10]:
# subset_columns = ['nombre_vacante', 'cargo', 'empresa', 'tipodocumentoempresa',
#        'estado_actual', 'tipo_de_vacante', 'programa_de_gobierno', 'ubicación', 'discapacidad',
#        'tipo_de_contrato', 'agente_aprobó', 'pertenece_hidrocarburos',
#        'tipo_de_proyecto_hidrocarburos', 'requiere_mano_de_obra_calificada',
#        'punto_atención', 'país']

# # Replace empty strings with NaN (optional)
# for column in subset_columns:
#     vacantes[column] = [str(i).lower() for i in vacantes[column]]
#     vacantes[column] = [str(i).strip() for i in vacantes[column]]

In [11]:
vacantes['fecha_registro'] = vacantes['fecha_registro'].fillna("")

# Function to handle different date formats
def parse_dates(date_str):
    if pd.isna(date_str) or date_str == "":  # Handle empty strings or NaN
        return pd.NaT
    
    date_str = str(date_str).strip()
    
    # Case 1: Excel serial number
    if date_str.isdigit():
        return pd.to_datetime(int(date_str), origin='1899-12-30', unit='D')

    # Case 2: DD-MM-YYYY format
    try:
        return pd.to_datetime(date_str, format="%d-%m-%Y")
    except ValueError:
        pass  # If it fails, try the next method
    
    # Case 3: Other datetime formats
    return pd.to_datetime(date_str, errors='coerce', dayfirst=True)  

# Convert date columns
vacantes['fecha_registro'] = vacantes['fecha_registro'].apply(parse_dates)

# Floor the date to remove time (ensures it's still a datetime object)
vacantes['fecha_registro'] = vacantes['fecha_registro'].dt.floor('D')

C:\Users\jober\AppData\Local\Temp\ipykernel_12664\360358374.py:21: UserWarning: Parsing dates in %Y-%m-%d %H:%M:%S format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  return pd.to_datetime(date_str, errors='coerce', dayfirst=True)


In [12]:
# Extracting the relevant data to work with
vacantes = vacantes[['código_proceso', 'nombre_vacante', 'cargo', '#_postulados', 'empresa',
       'tipodocumentoempresa', 'numerodocumentoempresa', 'fecha_registro',
       'fecha_vencimiento', 'estado_actual', 'tipo_de_vacante',
       'puestos_de_trabajo', 'tipo_de_contrato', 'agente_aprobó',
       'mes', 'año', 'punto_atención', 'país']]

In [13]:
# The new companies for this months are
today = datetime.today()

if today.month == 1:
    prev_month = 12
    prev_year = today.year - 1
else:
    prev_month = today.month - 1
    prev_year = today.year

In [14]:
# prev_month = 7 
# prev_year = 2025

In [15]:
# The filtered registries are: 
filter = (vacantes['mes'] == prev_month) & (vacantes['año'] == prev_year)

vacantes = vacantes[filter]
# vacantes.head()

In [16]:
vacantes['empresa'].count()

78

In [17]:
# Exporting the data
# vacantes.to_parquet(f'vacantes_2024', compression='zstd')
vacantes.to_parquet(f'vacantes_{prev_year}_{prev_month}.parquet', compression='zstd')
